<a href="https://colab.research.google.com/github/GitHubAman2004/stock_price_predictor/blob/fixed_class_imbalance/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
%pip install yfinance pandas

In [36]:
%pip install scikit-learn

In [37]:
import torch
dummy_x=torch.randn(32,60,5)

In [38]:
import torch
import torch.nn as nn
cnn_input=dummy_x.permute(0,2,1)
conv_layer=nn.Conv1d(in_channels=5,out_channels=16,kernel_size=3,padding=1)
cnn_output=conv_layer(cnn_input)
final_output=cnn_output.permute(0,2,1)
print("original shape", dummy_x.shape)
print("after cnn block", final_output.shape)

original shape torch.Size([32, 60, 5])
after cnn block torch.Size([32, 60, 16])


In [39]:
import torch
import torch.nn as nn
lstm_layer=nn.LSTM(input_size=16,hidden_size=16,batch_first=True)
lstm_output,(h_n,c_n)=lstm_layer(final_output)
print("shape entering lstm",final_output.shape)
print("shape exiting lstm",lstm_output.shape)

shape entering lstm torch.Size([32, 60, 16])
shape exiting lstm torch.Size([32, 60, 16])


In [40]:
import torch
import torch.nn as nn
attention_layer=nn.MultiheadAttention(embed_dim=32,num_heads=4,batch_first=True)

In [41]:
import torch
import torch.nn as nn

class HybridForecastingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Conv1d(in_channels=5, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
        self.fc = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.cnn(x))
        x = x.permute(0, 2, 1)

        x, _ = self.lstm(x)
        x, _ = self.attention(x, x, x)

        last_day_features = x[:, -1, :]
        prediction = self.fc(last_day_features)

        return prediction

model = HybridForecastingModel()
dummy_x = torch.randn(32, 60, 5)
final_prediction = model(dummy_x)

print("Final prediction shape:", final_prediction.shape)

Final prediction shape: torch.Size([32, 1])


In [42]:
import yfinance as yf
import pandas as pd
ticker="AAPL"
print("Downloading data from {ticker}...")
df=yf.download(ticker,period='5y')
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

print("\n--- Data successfully downloaded! ---")
print("Total trading days:", len(df))
print("\nFirst 3 days of data:")
print(df.head(5))

/tmp/ipykernel_1148/2908926255.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download(ticker,period='5y')
[*********************100%***********************]  1 of 1 completed


--- Data successfully downloaded! ---
Total trading days: 1255

First 3 days of data:
Price             Open        High         Low       Close    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2021-05-26  123.754509  124.173652  123.228142  123.647285  56575900
2021-05-27  123.247629  124.417328  121.921965  122.116913  94625600
2021-05-28  122.399590  122.623786  121.405346  121.463829  71311100
2021-06-01  121.921978  122.185158  120.810762  121.142174  67637100
2021-06-02  121.142168  122.077929  120.917979  121.902473  59278900


In [43]:
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
sequence_length = 60
X = []
y = []
for i in range(sequence_length, len(scaled_data) - 1):
    X.append(scaled_data[i - sequence_length : i])
    today_close = scaled_data[i, 3]
    tomorrow_close = scaled_data[i + 1, 3]

    if tomorrow_close > today_close:
        y.append(1)
    else:
        y.append(0)
X_tensor = torch.tensor(np.array(X), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(1)

print("Dataset successfully refined!")
print("X shape (Input):", X_tensor.shape)
print("y shape (Target):", y_tensor.shape)

Dataset successfully refined!
X shape (Input): torch.Size([1194, 60, 5])
y shape (Target): torch.Size([1194, 1])


In [44]:
from torch.utils.data import TensorDataset,DataLoader
split_idx=int(len(X_tensor)*0.80)

X_train=X_tensor[:split_idx]
y_train=y_tensor[:split_idx]

X_test=X_tensor[split_idx:]
y_test=y_tensor[split_idx:]

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

batch_size=32

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=False)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

print("--- Data perfectly partitioned! ---")
print(f"Training samples: {len(X_train)} (Split into {len(train_loader)} batches of {batch_size})")
print(f"Testing samples: {len(X_test)} (Split into {len(test_loader)} batches of {batch_size})")

--- Data perfectly partitioned! ---
Training samples: 955 (Split into 30 batches of 32)
Testing samples: 239 (Split into 8 batches of 32)


In [45]:
import torch.optim as optim
import torch.nn as nn
total_samples=len(train_loader.dataset)
neg_count=0
pos_count=0
for _,batch_y in train_loader:
  pos_count+=(batch_y==1).sum().item()
  neg_count+=(batch_y==0).sum().item()
print(f"Positive samples: {pos_count}")
print(f"Negative samples: {neg_count}")
print(f"pos_weight value: {neg_count / pos_count:.2f}")
model=HybridForecastingModel()
pos_weight=torch.tensor([neg_count/pos_count])
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=10
print("Starting Advanced Training")

for epoch in range (epochs):
  total_tp=0;
  total_fp=0;
  total_tn=0;
  total_fn=0;
  running_loss=0.0
  for batch_X,batch_y in train_loader:
     optimizer.zero_grad()
     predictions=model(batch_X)
     loss=criterion(predictions,batch_y)
     loss.backward()
     optimizer.step()

     running_loss+=loss.item()
     predicted_classes=(predictions>0).float()


     total_tp += ((predicted_classes == 1) & (batch_y == 1)).sum().item()
     total_fp += ((predicted_classes == 1) & (batch_y == 0)).sum().item()
     total_fn += ((predicted_classes == 0) & (batch_y == 1)).sum().item()
     total_tn += ((predicted_classes == 0) & (batch_y == 0)).sum().item()

  epoch_loss=running_loss/len(train_loader)

  total_samples = total_tp + total_tn + total_fp + total_fn
  accuracy = (total_tp + total_tn) / total_samples

  precision = total_tp / (total_tp + total_fp + 1e-7)

  recall = total_tp / (total_tp + total_fn + 1e-7)

  f1_score = 2 * (precision * recall) / (precision + recall + 1e-7)

  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {accuracy*100:.1f}% | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1_score:.3f}")

print("--- Training Complete! ---")

Positive samples: 504
Negative samples: 451
pos_weight value: 0.89
Starting Advanced Training
Epoch [1/10] | Loss: 0.6930 | Acc: 51.5% | Precision: 0.543 | Recall: 0.512 | F1: 0.527
Epoch [2/10] | Loss: 0.6921 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [3/10] | Loss: 0.6921 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [4/10] | Loss: 0.6921 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [5/10] | Loss: 0.6921 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [6/10] | Loss: 0.6920 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [7/10] | Loss: 0.6920 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [8/10] | Loss: 0.6920 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [9/10] | Loss: 0.6920 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
Epoch [10/10] | Loss: 0.6920 | Acc: 52.8% | Precision: 0.528 | Recall: 1.000 | F1: 0.691
--- Training Complete! --